# LMA Phase 1 — Complete Pipeline (Scrape + Clean + Tokenize)

Runs on **Colab CPU** (no GPU needed). All data on Google Drive.

**Pipeline:**
1. Setup: mount Drive, unzip code, install deps
2. Download Hindi 80% from HuggingFace (C4 + Wikipedia)
3. Scrape Hindi + Nepali Wikisource (manual 20%)
4. Clean all data (downloaded + manual) for both languages
5. Merge → splits (98/1/1)
6. Train BPE tokenizers (Hindi + Nepali, 32k vocab each)
7. Encode → measure tokens → stats
8. Tokenization examples for report

**Before starting:** upload `lma_code.zip` to `MyDrive/LMA_Project/` in your Drive.

## 0. Setup: Mount Drive + unzip code + install deps

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
PROJ = '/content/drive/MyDrive/LMA_Project'
import os; os.makedirs(PROJ, exist_ok=True)
print('Drive mounted at', PROJ)

In [ ]:
# Unzip pipeline code (one-time)
import zipfile, os
zip_path = f'{PROJ}/lma_code.zip'
dest = '/content/lma_code'
if os.path.exists(zip_path):
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall('/content/')
    print(f'Unzipped to {dest}')
else:
    print(f'ERROR: {zip_path} not found. Upload lma_code.zip to Drive first!')

In [ ]:
import sys; sys.path.insert(0, '/content/lma_code')

# Quick import test
import common.clean, common.dedup, common.splits, common.stats, common.collect, common.langfilter
from hindi.tokenizer.tokenizer import Tokenizer as HiTok
from nepali.tokenizer.tokenizer import Tokenizer as NeTok
print('All imports OK — pipeline ready.')

In [ ]:
!pip install -q regex datasets
print('deps installed')

## 1. Download Hindi 80% from HuggingFace

Downloads Wikipedia (hi) + C4 (hi) via streaming. Target ~4 GB raw text.
This is the **downloaded** portion (tagged `source: downloaded`).

In [ ]:
# Download Hindi Wikipedia (~250 MB)
!cd /content/lma_code && python -m common.collect --lang hindi --sources wiki \
    --output-dir /content/drive/MyDrive/LMA_Project/hindi/raw/downloaded \
    --max-chars 1000000000 --shard-size 250000

In [ ]:
# Download Hindi C4 (~4 GB — this is the big one)
!cd /content/lma_code && python -m common.collect --lang hindi --sources mc4 \
    --output-dir /content/drive/MyDrive/LMA_Project/hindi/raw/downloaded \
    --max-chars 4000000000 --shard-size 250000

## 2. Scrape Wikisource (manual data for 20% requirement)

Scrapes **original-language** public-domain texts from Hindi and Nepali
Wikisource. These are NOT translations — they are original Hindi/Nepali
literature uploaded by the respective Wikimedia communities.

In [ ]:
# Wikisource scraper — shared functions
import json, os, re, time, urllib.parse, urllib.request

WIKISOURCE = {"hindi": "hi.wikisource.org", "nepali": "ne.wikisource.org"}

_RE_WIKI_TEMPLATE = re.compile(r"\{\{[^}]*\}\}")
_RE_WIKI_REF = re.compile(r"<ref[^>]*>.*?</ref>", re.S)
_RE_WIKI_COMMENT = re.compile(r"<!--[^>]*-->", re.S)
_RE_WIKI_HTML = re.compile(r"<[^>]+>")
_RE_WIKI_HEADINGS = re.compile(r"^={2,}\s*(.*?)\s*={2,}$", re.M)
_RE_WIKI_LIST = re.compile(r"^[\*#:;]+\s*", re.M)
_RE_WIKI_LINK = re.compile(r"\[\[(?:[^\|\]]*\|)?([^\]]*)\]\]")
_RE_WIKI_CATEG = re.compile(r"\[\[(?:Category|श्रेणी):[^\]]*\]\]")
_RE_WIKI_FILE = re.compile(r"\[\[(?:File|Image|चित्र):[^\]]*\]\]")
_RE_MULTI_SPACE = re.compile(r"[ \t]+")
_RE_BLANK_LINES = re.compile(r"\n{3,}")

def strip_wiki(wt):
    t = _RE_WIKI_COMMENT.sub("", wt)
    t = _RE_WIKI_REF.sub("", t)
    t = _RE_WIKI_TEMPLATE.sub("", t)
    t = _RE_WIKI_HTML.sub(" ", t)
    t = _RE_WIKI_CATEG.sub("", t)
    t = _RE_WIKI_FILE.sub("", t)
    t = _RE_WIKI_HEADINGS.sub(r"\1", t)
    t = _RE_WIKI_LIST.sub("", t)
    t = _RE_WIKI_LINK.sub(r"\1", t)
    t = _RE_MULTI_SPACE.sub(" ", t)
    t = _RE_BLANK_LINES.sub("\n\n", t)
    return t.strip()

def api_get(domain, params):
    url = f"https://{domain}/w/api.php?" + urllib.parse.urlencode({**params, "format": "json"})
    req = urllib.request.Request(url, headers={"User-Agent": "LMA-Project/1.0 (educational)"})
    with urllib.request.urlopen(req, timeout=30) as r:
        return json.loads(r.read().decode("utf-8"))

def list_pages(domain, max_pages):
    titles = []
    apfrom = None
    while len(titles) < max_pages:
        params = {"action": "query", "list": "allpages", "apnamespace": "0",
                  "aplimit": "500", "apfilterredir": "nonredirects"}
        if apfrom: params["apfrom"] = apfrom
        data = api_get(domain, params)
        pages = data.get("query", {}).get("allpages", [])
        titles.extend(p["title"] for p in pages)
        cont = data.get("query-continue", data.get("continue", {}))
        apfrom = cont.get("allpages", {}).get("apfrom") or cont.get("apfrom")
        if not apfrom: break
        time.sleep(0.5)
    return titles[:max_pages]

def get_text(domain, title):
    try:
        data = api_get(domain, {"action": "parse", "page": title, "prop": "wikitext"})
        wt = data.get("parse", {}).get("wikitext", {}).get("*", "")
        return strip_wiki(wt) if wt else None
    except:
        return None

print("Wikisource scraper ready.")

In [ ]:
# Scrape HINDI Wikisource
lang = "hindi"
domain = WIKISOURCE[lang]
outdir = f'/content/drive/MyDrive/LMA_Project/{lang}/raw/manual/wikisource'
os.makedirs(outdir, exist_ok=True)

print(f"Listing pages from {domain}...")
titles = list_pages(domain, 5000)
print(f"Found {len(titles)} pages. Scraping...")

f_out = open(os.path.join(outdir, f'{lang}_wikisource_0000.jsonl'), 'w', encoding='utf-8')
n_out = total_chars = 0
t0 = time.time()

for i, title in enumerate(titles):
    text = get_text(domain, title)
    if not text or len(text) < 50: continue
    rec = {"text": text, "source": "manual", "lang": lang, "dataset": "wikisource",
           "title": title, "n_chars": len(text), "n_bytes": len(text.encode('utf-8'))}
    f_out.write(json.dumps(rec, ensure_ascii=False) + '\n')
    n_out += 1; total_chars += len(text)
    if (i+1) % 100 == 0:
        print(f"  [{i+1}/{len(titles)}] saved {n_out}, {total_chars/1e6:.1f} MB", flush=True)
    time.sleep(0.2)

f_out.close()
print(f"HINDI Wikisource DONE: {n_out} pages, {total_chars/1e6:.1f} MB in {time.time()-t0:.1f}s")

In [ ]:
# Scrape NEPALI Wikisource
lang = "nepali"
domain = WIKISOURCE[lang]
outdir = f'/content/drive/MyDrive/LMA_Project/{lang}/raw/manual/wikisource'
os.makedirs(outdir, exist_ok=True)

print(f"Listing pages from {domain}...")
titles = list_pages(domain, 5000)
print(f"Found {len(titles)} pages. Scraping...")

f_out = open(os.path.join(outdir, f'{lang}_wikisource_0000.jsonl'), 'w', encoding='utf-8')
n_out = total_chars = 0
t0 = time.time()

for i, title in enumerate(titles):
    text = get_text(domain, title)
    if not text or len(text) < 50: continue
    rec = {"text": text, "source": "manual", "lang": lang, "dataset": "wikisource",
           "title": title, "n_chars": len(text), "n_bytes": len(text.encode('utf-8'))}
    f_out.write(json.dumps(rec, ensure_ascii=False) + '\n')
    n_out += 1; total_chars += len(text)
    if (i+1) % 100 == 0:
        print(f"  [{i+1}/{len(titles)}] saved {n_out}, {total_chars/1e6:.1f} MB", flush=True)
    time.sleep(0.2)

f_out.close()
print(f"NEPALI Wikisource DONE: {n_out} pages, {total_chars/1e6:.1f} MB in {time.time()-t0:.1f}s")

In [ ]:
# Verify scraped text is genuine Devanagari (not English/transliteration)
import json, glob
for lang in ['hindi', 'nepali']:
    print(f'\n===== {lang.upper()} WIKISOURCE SAMPLE =====')
    files = glob.glob(f'/content/drive/MyDrive/LMA_Project/{lang}/raw/manual/wikisource/*.jsonl')
    if not files:
        print('  no files yet'); continue
    with open(files[0]) as f:
        for i, line in enumerate(f):
            if i >= 3: break
            rec = json.loads(line)
            print(f'  title: {rec.get("title", "?")}')
            print(f'  text: {rec["text"][:200]}')
            print()

## 3. Clean all data (downloaded + manual)

- Downloaded: strict filter (0.55 Devanagari, 80 chars min)
- Manual OCR: loose filter (0.20 Devanagari, 15 chars min) — OCR has noise
- Manual Wikisource: medium filter (0.40 Devanagari, 40 chars min)

In [ ]:
# Clean HINDI DOWNLOADED (C4 + Wikipedia shards)
import glob, subprocess, sys, os
out = '/content/drive/MyDrive/LMA_Project/hindi/processed/clean'
for f in sorted(glob.glob('/content/drive/MyDrive/LMA_Project/hindi/raw/downloaded/*.jsonl')):
    print('cleaning', os.path.basename(f))
    subprocess.run([sys.executable, '-m', 'common.clean', '--lang', 'hindi',
                    '--input', f, '--output-dir', out,
                    '--source-tag', 'downloaded', '--shard-size', '250000',
                    '--min-chars', '80'], cwd='/content/lma_code')
print('Hindi downloaded cleaning done')

In [ ]:
# Clean HINDI MANUAL (webcrawl + wikisource)
import glob, subprocess, sys, os, shutil

out = '/content/drive/MyDrive/LMA_Project/hindi/processed/clean_manual'
if os.path.isdir(out): shutil.rmtree(out)

for f in sorted(glob.glob('/content/drive/MyDrive/LMA_Project/hindi/raw/manual/**/*.jsonl', recursive=True)):
    print('cleaning manual', os.path.basename(f))
    # Use looser filter for wikisource (may have some English headers)
    subprocess.run([sys.executable, '-m', 'common.clean', '--lang', 'hindi',
                    '--input', f, '--output-dir', out,
                    '--source-tag', 'manual',
                    '--min-chars', '40',
                    '--min-devanagari-ratio', '0.40'], cwd='/content/lma_code')
print('Hindi manual cleaning done')

In [ ]:
# Clean NEPALI DOWNLOADED (22GB train + 5GB test — if not already done)
# Check if already cleaned
import os, glob
existing = glob.glob('/content/drive/MyDrive/LMA_Project/nepali/processed/clean/nepali_clean_*.jsonl')
if len(existing) > 1:
    print(f'Nepali downloaded already cleaned ({len(existing)} shards) — skipping')
else:
    print('Cleaning Nepali downloaded...')
    !cd /content/lma_code && python -m common.clean --lang nepali \
        --input /content/drive/MyDrive/LMA_Project/nepali/raw/downloaded/nepali_train.jsonl \
        --output-dir /content/drive/MyDrive/LMA_Project/nepali/processed/clean \
        --source-tag downloaded --shard-size 250000 --min-chars 80
    !cd /content/lma_code && python -m common.clean --lang nepali \
        --input /content/drive/MyDrive/LMA_Project/nepali/raw/downloaded/nepali_test.jsonl \
        --output-dir /content/drive/MyDrive/LMA_Project/nepali/processed/clean \
        --source-tag downloaded --shard-size 250000 --min-chars 80

In [ ]:
# Clean NEPALI MANUAL (OCR + scraped + wikisource) with LOOSE filters
# OCR text has English/ISSN/numbers mixed in — needs 0.20 ratio + 15 char min
import glob, subprocess, sys, os, shutil

out = '/content/drive/MyDrive/LMA_Project/nepali/processed/clean_manual'
if os.path.isdir(out): shutil.rmtree(out)

manual_dirs = [
    '/content/drive/MyDrive/LMA_Project/nepali/raw/manual/extracted',
    '/content/drive/MyDrive/LMA_Project/nepali/raw/manual/sahityaras/extracted',
    '/content/drive/MyDrive/LMA_Project/nepali/raw/manual/cdc/extracted',
    '/content/drive/MyDrive/LMA_Project/nepali/raw/manual/national_library/ocr',
    '/content/drive/MyDrive/LMA_Project/nepali/raw/manual/national_library/extracted',
    '/content/drive/MyDrive/LMA_Project/nepali/raw/manual/wikisource',
]
for d in manual_dirs:
    if not os.path.isdir(d): continue
    for f in glob.glob(os.path.join(d, '*.jsonl')):
        print('cleaning', os.path.basename(f))
        subprocess.run([sys.executable, '-m', 'common.clean', '--lang', 'nepali',
                        '--input', f, '--output-dir', out,
                        '--source-tag', 'manual',
                        '--min-chars', '15',
                        '--min-devanagari-ratio', '0.20'], cwd='/content/lma_code')
print('Nepali manual cleaning done')

In [ ]:
# Check all cleaned sizes
import os, glob
PROJ = '/content/drive/MyDrive/LMA_Project'
for lang in ['hindi', 'nepali']:
    print(f'\n===== {lang.upper()} =====')
    for subdir in ['processed/clean', 'processed/clean_manual']:
        d = f'{PROJ}/{lang}/{subdir}'
        if not os.path.isdir(d): continue
        total = 0; files = 0
        for f in glob.glob(os.path.join(d, '*.jsonl')):
            total += os.path.getsize(f); files += 1
        print(f'  {subdir}: {files} files, {total/1e6:.1f} MB')

## 4. Merge downloaded + manual → splits (98/1/1)

In [ ]:
# Merge downloaded + manual cleaned into one dir (symlinks)
import os, glob
for lang in ['hindi', 'nepali']:
    merged = f'/content/drive/MyDrive/LMA_Project/{lang}/processed/merged'
    os.makedirs(merged, exist_ok=True)
    # clear old symlinks
    for f in glob.glob(os.path.join(merged, '*.jsonl')):
        os.remove(f)
    for src_dir in [f'/content/drive/MyDrive/LMA_Project/{lang}/processed/clean',
                    f'/content/drive/MyDrive/LMA_Project/{lang}/processed/clean_manual']:
        if os.path.isdir(src_dir):
            for f in glob.glob(os.path.join(src_dir, '*.jsonl')):
                dst = os.path.join(merged, os.path.basename(f))
                if not os.path.exists(dst):
                    os.symlink(os.path.abspath(f), dst)
    print(lang, 'merged:', len(os.listdir(merged)), 'files')

In [ ]:
# Create train/val/test splits (98/1/1, doc-level, seeded)
for lang in ['hindi', 'nepali']:
    inp = f'/content/drive/MyDrive/LMA_Project/{lang}/processed/merged'
    out = f'/content/drive/MyDrive/LMA_Project/{lang}/splits'
    !cd /content/lma_code && python -m common.splits --lang {lang} --input {inp} \
        --output-dir {out} --train 0.98 --val 0.01 --test 0.01 --seed 17

## 5. Train BPE tokenizers (32k vocab, ~2-5 min each)

Trains on an 80M-char subsample of the training split. The trainer uses
incremental pair-counting with a max-heap (fast — minutes, not weeks).

In [ ]:
# Train Hindi BPE
!cd /content/lma_code && python -m hindi.tokenizer.train --language hindi \
    --data /content/drive/MyDrive/LMA_Project/hindi/splits/hindi_train.jsonl \
    --vocab-size 32000 --char-budget 80000000 \
    --output /content/drive/MyDrive/LMA_Project/hindi/tokenizer/hindi_tokenizer.json

In [ ]:
# Train Nepali BPE
!cd /content/lma_code && python -m nepali.tokenizer.train --language nepali \
    --data /content/drive/MyDrive/LMA_Project/nepali/splits/nepali_train.jsonl \
    --vocab-size 32000 --char-budget 80000000 \
    --output /content/drive/MyDrive/LMA_Project/nepali/tokenizer/nepali_tokenizer.json

## 6. Encode full corpora → measure tokens + manual% + stats

In [ ]:
# Corpus stats + token counts + manual/downloaded split + unknown rate
for lang in ['hindi', 'nepali']:
    splits = f'/content/drive/MyDrive/LMA_Project/{lang}/splits'
    tok = f'/content/drive/MyDrive/LMA_Project/{lang}/tokenizer/{lang}_tokenizer.json'
    heldout = f'{splits}/{lang}_heldout.jsonl'
    out = f'/content/drive/MyDrive/LMA_Project/{lang}/stats/{lang}_stats.json'
    !cd /content/lma_code && python -m common.stats --lang {lang} --input {splits} \
        --output {out} --tokenizer {tok} --heldout {heldout}

In [ ]:
# Display the stats
import json, os
for lang in ['hindi', 'nepali']:
    sp = f'/content/drive/MyDrive/LMA_Project/{lang}/stats/{lang}_stats.json'
    if os.path.exists(sp):
        print(f'\n===== {lang.upper()} STATS =====')
        s = json.load(open(sp))
        for k, v in s.items():
            if k != 'source_breakdown':
                print(f'  {k}: {v}')
        if 'source_breakdown' in s:
            print('  source_breakdown:')
            for kk, vv in s['source_breakdown'].items():
                print(f'    {kk}: {vv}')

## 7. Tokenization examples (for the report)

In [ ]:
import sys, os
sys.path.insert(0, '/content/lma_code')
from nepali.tokenizer.tokenizer import Tokenizer as NeTok
from hindi.tokenizer.tokenizer import Tokenizer as HiTok

examples = {
  'hindi': ['नमस्ते दुनिया, यह एक परीक्षण वाक्य है।',
            'भारत एक विशाल और विविध देश है।'],
  'nepali': ['नमस्ते संसार, यो एक परीक्षण वाक्य हो।',
             'नेपाल एक सुन्दर हिमालयको देश हो।'],
}
for lang, toks in [('hindi', HiTok), ('nepali', NeTok)]:
    tpath = f'/content/drive/MyDrive/LMA_Project/{lang}/tokenizer/{lang}_tokenizer.json'
    if not os.path.exists(tpath):
        print(f'{lang}: tokenizer not found'); continue
    tok = toks.load(tpath)
    print(f'\n===== {lang.upper()} tokenization examples =====')
    for ex in examples[lang]:
        ids = tok.encode(ex)
        print(f'text: {ex}')
        print(f'ids : {ids}')
        print(f'dec : {tok.decode(ids)}')
        print(f'round_trip: {tok.verify_round_trip(ex)}')
        print(f'stats: {tok.tokenization_stats(ex)}')
        print()

## 8. Done!

All artifacts on Drive:
- Cleaned: `LMA_Project/{lang}/processed/clean/` (downloaded)
- Cleaned manual: `LMA_Project/{lang}/processed/clean_manual/`
- Splits: `LMA_Project/{lang}/splits/`
- Tokenizer: `LMA_Project/{lang}/tokenizer/{lang}_tokenizer.json`
- Stats: `LMA_Project/{lang}/stats/{lang}_stats.json`

**Next:** Copy the stats JSONs and tokenizer JSONs to the repo on your Mac
for git. Large data stays on Drive. Paste the stats output to fill in
the Phase 1 report.